# 05. Evaluation and visualization

목표: 04의 immutable certified feature artifact에서 표준 open-set 지표와 인증 coverage를 계산하고, 작은 표/그림을 저장한 뒤 run을 완료합니다. 성공 기준은 registered의 DIR/FNIR와 non-mated의 FPIR가 분리되고 결과/그림 hash가 log에 남는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 04의 가장 최근 `completed` attempt만 읽습니다. 중단되면 05 전체를 다시 실행해 새 attempt를 만듭니다. `run.complete()` 후에는 immutable이므로 같은 run을 수정하지 말고, 평가 설정을 바꿀 때는 00부터 새 run을 만드십시오. 완료 run의 결과 확인은 읽기 전용으로만 수행합니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import RunStore, resolve_active_run

EXECUTE_STAGE = False
RUN_ROOT = PROJECT_ROOT / 'runs'
RUN_DIR = resolve_active_run(RUN_ROOT)


## Plan

- Resolve the latest completed 04 attempt without scanning arbitrary CSVs.
- Compute standard mated DIR/FNIR and non-mated FPIR plus certification coverage/defer/fallback.
- Save concise JSON/CSV and two readable figures, then mark the run completed.


In [ ]:
def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for path in sorted(attempts.glob('A*/phase_manifest.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}.')
    return max(completed)

preflight = {'execute_stage': EXECUTE_STAGE, 'run_dir_resolved': str(RUN_DIR),
             'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file())}
preflight


## Execute, visualize, and finalize

registered와 unknown을 하나의 accuracy로 합치지 않습니다. `known_unknown`과 `unknown_unknown`도 표와 그림에서 분리해 보고합니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from research.evaluation import open_set_identification_metrics, rank_at_k
    from research.runtime.hashing import sha256_file
    from research.search.open_set import summarize_certified_search_features

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    run.verify_phase_artifacts('04_probe_search_and_certification')
    search_attempt = latest_completed_attempt(RUN_DIR, '04_probe_search_and_certification')
    search_suffix = f'A{search_attempt:03d}'
    features_path = RUN_DIR / 'artifacts' / '04_probe_search_and_certification' / f'certified_features_{search_suffix}.csv'
    if not features_path.is_file():
        raise FileNotFoundError(features_path)
    with run.phase('05_evaluation_and_visualization') as phase:
        features = pd.read_csv(features_path)
        decision_column = 'final_decision' if 'final_decision' in features.columns else 'certified_decision'
        evaluation = features.copy()
        evaluation['accepted'] = evaluation[decision_column].eq('accept')
        predicted_identity_column = ('final_identity' if decision_column == 'final_decision' and 'final_identity' in evaluation.columns
                                     else 'certified_identity' if 'certified_identity' in evaluation.columns else 'top1_identity')
        metrics = {
            'source_search_attempt': search_suffix,
            'open_set': open_set_identification_metrics(evaluation, predicted_identity_column=predicted_identity_column),
            'certification': summarize_certified_search_features(features),
        }
        if 'ranked_identities' in evaluation.columns:
            evaluation['ranked_identities'] = evaluation['ranked_identities'].map(
                lambda value: json.loads(value) if isinstance(value, str) else value
            )
            metrics['rank_at_1'] = rank_at_k(evaluation, k=1)
        by_probe = (evaluation.groupby('probe_type', sort=True)
                    .agg(count=('probe_type', 'size'), accept_rate=('accepted', 'mean'), mean_top1_score=('top1_score', 'mean'))
                    .reset_index())
        suffix = f'A{phase.attempt:03d}'
        metrics_source = phase.attempt_dir / f'evaluation_metrics_{suffix}.json'
        table_source = phase.attempt_dir / f'probe_type_summary_{suffix}.csv'
        metrics_source.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
        by_probe.to_csv(table_source, index=False)
        phase.publish_artifact(metrics_source)
        phase.publish_artifact(table_source)

        decision_counts = pd.crosstab(evaluation['probe_type'], evaluation[decision_column])
        fig, ax = plt.subplots(figsize=(7, 4))
        decision_counts.plot(kind='bar', stacked=True, ax=ax)
        ax.set(title='Certified/final decisions by probe type', xlabel='Probe type', ylabel='Count')
        ax.legend(title='Decision'); fig.tight_layout()
        figure_source = phase.attempt_dir / f'decisions_by_probe_{suffix}.png'
        fig.savefig(figure_source, dpi=160); plt.close(fig)
        figure_destination = RUN_DIR / 'figures' / figure_source.name
        if figure_destination.exists():
            raise FileExistsError(figure_destination)
        os.replace(figure_source, figure_destination)
        phase.record('figure_published', path=str(figure_destination.relative_to(RUN_DIR)), sha256=sha256_file(figure_destination))

        fig, ax = plt.subplots(figsize=(7, 4))
        for probe_type, group in evaluation.groupby('probe_type', sort=True):
            ax.hist(group['top1_score'].astype(float), bins=25, alpha=0.45, label=str(probe_type))
        ax.set(title='Top-1 score distribution', xlabel='Cosine score', ylabel='Count'); ax.legend(); fig.tight_layout()
        score_source = phase.attempt_dir / f'top1_score_distribution_{suffix}.png'
        fig.savefig(score_source, dpi=160); plt.close(fig)
        score_destination = RUN_DIR / 'figures' / score_source.name
        if score_destination.exists():
            raise FileExistsError(score_destination)
        os.replace(score_source, score_destination)
        phase.record('figure_published', path=str(score_destination.relative_to(RUN_DIR)), sha256=sha256_file(score_destination))
        phase.record_counts(rows=len(evaluation), figures=2)
    run.complete()
    result = {'status': 'completed', 'run_id': run.run_id, **metrics}
result


## Final check

`run_manifest.json`의 status와 `COMPLETED`, phase manifests, events JSONL, artifact/figure hash를 확인합니다. 논문용 복사본은 이 immutable run을 source로 삼고 원본 결과를 직접 수정하지 않습니다.
